# 02 — Iterative Pursuit

The intercept solution from Notebook 01 requires knowing the target's future position — which means trusting that the target will keep its heading and speed. 

What if you cannot predict that? Or simply do not want to?

**Pure pursuit** takes a different approach: at every moment, re-compute the bearing to the target's *current* position and steer directly toward it. No prediction. No math beyond what you already know. Just: look, aim, move, repeat.

It sounds reasonable. It even works — sometimes. But the path it produces is curved, the flight time is longer, and under certain conditions the pursuer never catches the target at all.

This notebook simulates both approaches side by side so you can see exactly what prediction buys you.

In [ ]:
import math

def compute_bearing(p1, p2):
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def destination_point(origin, bearing_deg, distance_km):
    R = 6371.0
    d   = distance_km / R
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])
    lat2 = math.asin(math.sin(lat1)*math.cos(d) + math.cos(lat1)*math.sin(d)*math.cos(brg))
    lon2 = lon1 + math.atan2(math.sin(brg)*math.sin(d)*math.cos(lat1),
                              math.cos(d) - math.sin(lat1)*math.sin(lat2))
    return [math.degrees(lon2), math.degrees(lat2)]

# Module scenario
shooter_pos    = [-98.49, 33.91]
target_pos     = [-101.0, 36.5]
target_speed   = 300
target_heading = 135
shooter_speed  = 600

print("Ready.")

## 1. The Pure Pursuit Loop

Pure pursuit is a simulation, not a formula. At each time step:

1. Compute the bearing from the pursuer's current position to the target's current position
2. Move the pursuer one step in that direction
3. Move the target one step along its heading
4. Check if the pursuer is within capture distance

Repeat until caught or the step limit is reached.

**Time step size** is a design choice with a real trade-off:
- Too large → the pursuer overshoots the target on close approach; path accuracy degrades
- Too small → more computation, but also more accurate curves

A good rule of thumb: the step distance (`speed × dt`) should be well under the capture radius. For our scenario, steps of `dt = 0.01 h` (36 seconds) move the pursuer ~6 km and the target ~3 km per step — small enough to be accurate.

In [ ]:
def simulate_pursuit(shooter_pos, target_pos, target_heading, target_speed,
                     shooter_speed, dt=0.01, max_steps=2000, capture_km=5.0):
    """
    Simulate pure pursuit: pursuer always steers toward target's current position.

    Parameters
    ----------
    dt          : float, hours per time step
    max_steps   : int, step limit (prevents infinite loops)
    capture_km  : float, km — distance at which capture is declared

    Returns
    -------
    dict with:
        pursuer_path  : list of [lon, lat] — pursuer positions at each step
        target_path   : list of [lon, lat] — target positions at each step
        captured      : bool
        time_elapsed  : float, hours
        steps         : int
    """
    pursuer  = list(shooter_pos)
    target   = list(target_pos)
    p_path   = [list(pursuer)]
    t_path   = [list(target)]

    for step in range(max_steps):
        dist = haversine_km(pursuer, target)
        if dist <= capture_km:
            return {
                "pursuer_path": p_path,
                "target_path":  t_path,
                "captured":     True,
                "time_elapsed": step * dt,
                "steps":        step,
            }

        # Pursuer steers directly at current target position
        bearing = compute_bearing(pursuer, target)
        pursuer = destination_point(pursuer, bearing, shooter_speed * dt)

        # Target advances along its fixed heading
        target  = destination_point(target, target_heading, target_speed * dt)

        p_path.append(list(pursuer))
        t_path.append(list(target))

    return {
        "pursuer_path": p_path,
        "target_path":  t_path,
        "captured":     False,
        "time_elapsed": max_steps * dt,
        "steps":        max_steps,
    }


# Run the simulation
sim = simulate_pursuit(
    shooter_pos, target_pos, target_heading, target_speed, shooter_speed
)

status = "CAPTURED" if sim["captured"] else "NOT CAPTURED"
print(f"Result:         {status}")
print(f"Steps run:      {sim['steps']}")
print(f"Time elapsed:   {sim['time_elapsed']:.3f} h  ({sim['time_elapsed']*60:.1f} min)")
print(f"Path length:    {len(sim['pursuer_path'])} positions recorded")

## 2. Visualizing the Pursuit Curve

The pursuit path is not a straight line. Plot it alongside the target's path and the straight-line intercept solution from Notebook 01 and the shape becomes clear: the pursuer chases the target's tail, tracing a curve that converges from behind.

In [ ]:
from ipyleaflet import Map, GeoJSON

def fc(features): return {"type": "FeatureCollection", "features": features}
def line(coords, props={}): return {"type":"Feature","geometry":{"type":"LineString","coordinates":coords},"properties":props}
def point(coord, props={}): return {"type":"Feature","geometry":{"type":"Point","coordinates":coord},"properties":props}

# Intercept solution (straight line) from Notebook 01
def find_intercept_time(s_pos, t_pos, t_hdg, t_spd, s_spd, t_max=10.0, tol=1e-6):
    def f(t):
        future = destination_point(t_pos, t_hdg, t_spd * t)
        return haversine_km(s_pos, future) - s_spd * t
    if f(t_max) > 0: return None
    lo, hi = 0.0, t_max
    for _ in range(60):
        mid = (lo + hi) / 2
        (lo if f(mid) > 0 else hi).__class__   # dummy — just use assignment below
        if f(mid) > 0: lo = mid
        else: hi = mid
        if hi - lo < tol: break
    return (lo + hi) / 2

t_int = find_intercept_time(shooter_pos, target_pos, target_heading, target_speed, shooter_speed)
intercept_pt = destination_point(target_pos, target_heading, target_speed * t_int)

# Subsample pursuit path for map performance
step = max(1, len(sim["pursuer_path"]) // 200)
pursuer_coords = sim["pursuer_path"][::step]
target_coords  = sim["target_path"][::step]

m = Map(center=(35.5, -99.5), zoom=6)

# Target path
m.add(GeoJSON(data=fc([line(target_coords)]),
              style={"color": "#457b9d", "weight": 1.5, "dashArray": "5"}))
# Pursuit curve (red)
m.add(GeoJSON(data=fc([line(pursuer_coords)]),
              style={"color": "#e63946", "weight": 2}))
# Intercept straight line (green)
m.add(GeoJSON(data=fc([line([shooter_pos, intercept_pt])]),
              style={"color": "#2a9d8f", "weight": 2, "dashArray": "6"}))

# Key points
m.add(GeoJSON(data=fc([point(shooter_pos)]),
              point_style={"radius": 7, "color": "#e63946", "fillOpacity": 1.0}))
m.add(GeoJSON(data=fc([point(target_pos)]),
              point_style={"radius": 7, "color": "#457b9d", "fillOpacity": 1.0}))
m.add(GeoJSON(data=fc([point(intercept_pt)]),
              point_style={"radius": 6, "color": "#2a9d8f", "fillOpacity": 1.0}))

print("Red curve  = pure pursuit path")
print("Green line = constant-velocity intercept (straight)")
print("Blue dashes = target path")
m

The red curve bends toward the target and converges from behind. The green line goes straight to the intercept point. Both result in a catch — but the curved pursuit path is longer and takes more time.

## 3. Measuring the Cost of Pure Pursuit

How much extra distance and time does pure pursuit require compared to the straight intercept? The answer depends on the geometry: a target heading toward the pursuer costs little; a target heading away costs a lot.

In [ ]:
def path_length_km(path):
    """Total distance along a list of [lon, lat] positions."""
    return sum(haversine_km(path[i], path[i+1]) for i in range(len(path) - 1))


# Compare pursuit vs intercept across target headings
intercept_tof    = t_int * 60                         # minutes
intercept_dist   = haversine_km(shooter_pos, intercept_pt)

pursuit_tof      = sim["time_elapsed"] * 60
pursuit_dist     = path_length_km(sim["pursuer_path"])

print("=== Baseline scenario (target heading 135°, incoming SE) ===")
print(f"{'':30} {'Intercept':>12}  {'Pursuit':>10}  {'Δ':>8}")
print("-" * 66)
print(f"{'Time of flight (min)':<30} {intercept_tof:>11.1f}   {pursuit_tof:>9.1f}  {pursuit_tof - intercept_tof:>+7.1f}")
print(f"{'Distance flown (km)':<30} {intercept_dist:>11.1f}   {pursuit_dist:>9.1f}  {pursuit_dist - intercept_dist:>+7.1f}")
print(f"{'Captured':<30} {'yes':>12}   {'yes' if sim['captured'] else 'NO':>9}")
print()

# Sweep target headings to show when pursuit fails
print("Pure pursuit success vs. target heading (shooter 600, target 300 km/h):")
print(f"  {'Heading':>9}  {'Captured':>10}  {'TOF (min)':>12}  {'Path (km)':>12}")
print("  " + "-" * 48)

for hdg in range(0, 360, 30):
    s = simulate_pursuit(shooter_pos, target_pos, hdg, target_speed, shooter_speed,
                         dt=0.01, max_steps=3000)
    tof_str  = f"{s['time_elapsed']*60:.1f}" if s["captured"] else "—"
    dist_str = f"{path_length_km(s['pursuer_path']):.1f}" if s["captured"] else "—"
    cap_str  = "yes" if s["captured"] else "NO"
    print(f"  {hdg:>8}°   {cap_str:>9}   {tof_str:>11}   {dist_str:>11}")

## 4. The Effect of Step Size

Step size controls the accuracy of the simulation. A large `dt` means the pursuer takes big leaps and the bearing is only recomputed infrequently — the curve becomes jagged and the capture time inflates. A small `dt` gives a smoother, more accurate path but takes more steps to compute.

The table below shows how the simulated capture time changes with `dt` for the same scenario.

In [ ]:
dt_values = [0.1, 0.05, 0.02, 0.01, 0.005, 0.001]

print(f"{'dt (h)':>10}  {'Step dist (km)':>16}  {'TOF (min)':>12}  {'Steps':>8}  {'Captured':>10}")
print("-" * 64)
for dt in dt_values:
    max_s = int(5 / dt)   # cap at 5 simulated hours
    s = simulate_pursuit(shooter_pos, target_pos, target_heading, target_speed,
                         shooter_speed, dt=dt, max_steps=max_s)
    step_dist = shooter_speed * dt
    tof_str = f"{s['time_elapsed']*60:.2f}" if s["captured"] else "—"
    cap_str = "yes" if s["captured"] else "NO"
    print(f"{dt:>10.3f}   {step_dist:>14.1f} km  {tof_str:>11}   {s['steps']:>7}   {cap_str:>9}")

The capture time converges as `dt` decreases — that convergence is a sign the simulation is numerically stable. The `dt=0.01` value used throughout this notebook is a reasonable balance: accurate enough that the result is meaningful, fast enough to run interactively.

---

## Exercise A — Failure Modes

Set `shooter_speed = 250` and `target_speed = 300` (target is faster). Run `simulate_pursuit` for target headings `0°`, `90°`, `135°`, `180°`, and `270°`. 

For which headings does pure pursuit fail? For which does it still succeed? Explain in plain English why a slower pursuer can still catch a faster target under certain geometric conditions.

In [ ]:
test_headings = [0, 90, 135, 180, 270]

results = []
print(f"  {'Heading':>9}  {'Captured':>10}  {'TOF (min)':>12}")
print('  ' + '-' * 36)
for hdg in test_headings:
    s = simulate_pursuit(
        shooter_pos,
        target_pos,
        hdg,
        target_speed=300,
        shooter_speed=250,
        dt=0.01,
        max_steps=5000,
    )
    tof_str = f"{s['time_elapsed']*60:.1f}" if s['captured'] else '—'
    print(f"  {hdg:>8}°   {('yes' if s['captured'] else 'NO'):>9}   {tof_str:>11}")
    results.append((hdg, s['captured']))

print()
print('A slower pursuer still succeeds on headings that bring the target across or toward the shooter,')
print('because geometry creates a positive closing component even when raw speed is lower.')

## Exercise B — Map Multiple Pursuit Curves

Run `simulate_pursuit` for four different target headings (`45°`, `135°`, `225°`, `315°`) against the same shooter. Plot all four pursuit paths on a single ipyleaflet map using a different color per heading. Add the target start position and the shooter position as markers.

Which curves are shortest? Which are longest? Does the shape of the curve tell you anything about the geometry?

In [ ]:
headings_to_plot = [45, 135, 225, 315]
colors = ['#e63946', '#2a9d8f', '#e9c46a', '#457b9d']

multi_map = Map(center=(35.5, -99.5), zoom=6)
multi_map.add(GeoJSON(
    data=fc([point(shooter_pos), point(target_pos)]),
    point_style={'radius': 8, 'color': '#222', 'fillOpacity': 0.9},
))

curve_stats = []
for hdg, color in zip(headings_to_plot, colors):
    sim_case = simulate_pursuit(
        shooter_pos,
        target_pos,
        hdg,
        target_speed,
        shooter_speed,
        dt=0.01,
        max_steps=3000,
    )
    step = max(1, len(sim_case['pursuer_path']) // 200)
    multi_map.add(GeoJSON(
        data=fc([line(sim_case['pursuer_path'][::step], {'heading': hdg})]),
        style={'color': color, 'weight': 2.5},
    ))
    curve_stats.append({
        'heading': hdg,
        'captured': sim_case['captured'],
        'tof_min': sim_case['time_elapsed'] * 60,
        'path_km': path_length_km(sim_case['pursuer_path']),
    })

print(f"{'Heading':>8}  {'Captured':>10}  {'TOF (min)':>10}  {'Path (km)':>10}")
print('-' * 48)
for row in curve_stats:
    print(
        f"{row['heading']:>7}°  "
        f"{('yes' if row['captured'] else 'NO'):>10}  "
        f"{row['tof_min']:>9.1f}  {row['path_km']:>9.1f}"
    )

shortest = min(curve_stats, key=lambda row: row['tof_min'])
longest = max(curve_stats, key=lambda row: row['tof_min'])
print()
print(f"Shortest pursuit curve: heading {shortest['heading']}°")
print(f"Longest pursuit curve:  heading {longest['heading']}°")
multi_map

## Exercise C — Pursuit vs. Intercept Time Budget

For the baseline scenario, the intercept solution takes less time than pure pursuit. How much less depends on target heading. 

For each heading in `range(0, 360, 15)`:
1. Run `simulate_pursuit` and record the pursuit time (or `None` if it fails)
2. Run `find_intercept_time` and record the intercept time (or `None`)
3. Compute the time saved by using intercept over pursuit

Print a table and identify: which heading maximizes the time savings? Which heading makes pursuit and intercept nearly equivalent?

In [ ]:
rows = []
for hdg in range(0, 360, 15):
    pursuit = simulate_pursuit(
        shooter_pos,
        target_pos,
        hdg,
        target_speed=300,
        shooter_speed=600,
        dt=0.01,
        max_steps=4000,
    )
    pursuit_t = pursuit['time_elapsed'] * 60 if pursuit['captured'] else None
    intercept_t = find_intercept_time(shooter_pos, target_pos, hdg, 300, 600)
    intercept_min = intercept_t * 60 if intercept_t is not None else None
    if pursuit_t is not None and intercept_min is not None:
        saved = pursuit_t - intercept_min
    else:
        saved = None
    rows.append({
        'heading': hdg,
        'pursuit_min': pursuit_t,
        'intercept_min': intercept_min,
        'saved_min': saved,
    })

print(f"{'Heading':>8}  {'Pursuit':>10}  {'Intercept':>11}  {'Saved':>9}")
print('-' * 46)
for row in rows:
    p = f"{row['pursuit_min']:.1f}" if row['pursuit_min'] is not None else '—'
    i = f"{row['intercept_min']:.1f}" if row['intercept_min'] is not None else '—'
    s = f"{row['saved_min']:.1f}" if row['saved_min'] is not None else '—'
    print(f"{row['heading']:>7}°  {p:>10}  {i:>11}  {s:>9}")

valid = [row for row in rows if row['saved_min'] is not None]
best = max(valid, key=lambda row: row['saved_min'])
closest = min(valid, key=lambda row: abs(row['saved_min']))
print()
print(f"Maximum time savings: heading {best['heading']}°  ({best['saved_min']:.1f} min faster)")
print(f"Nearly equivalent:    heading {closest['heading']}°  ({closest['saved_min']:.1f} min difference)")

---

## Check Your Understanding

A student runs `simulate_pursuit` with `dt=0.1` and gets a capture time of 28 minutes. They run it again with `dt=0.001` and get 24 minutes. They conclude the simulation is broken because it gives different answers.

**Question:** Is the simulation broken? Explain why the two results differ, which one is more accurate, and how you would determine when `dt` is small enough that further reduction stops mattering.

The simulation is not broken; it is a numerical approximation, so a larger `dt` takes coarse jumps and overestimates the path. The smaller `dt` is usually more accurate because it re-aims more often and follows the curved pursuit path more closely. The right test is convergence: keep shrinking `dt` until the capture time changes only by a negligible amount between runs.

---

## Next

In [03 — Visual Simulation](./03-Visual_Simulation.ipynb), we bring both strategies to life — click to place the shooter and target, set the motion, and watch intercept and pursuit animate side by side on a live map.